#Approach
Get movies rated by user
Multiply movie vector by rating
Sum all vectors
Normalize by total rating

In [1]:
from pyspark.sql import SparkSession                # create Spark session
from pyspark.sql.functions import col, split        # column operations
from pyspark.sql.functions import expr              # SQL expressions
from pyspark.sql.functions import collect_list      # aggregate lists
from pyspark.sql.functions import avg               # compute averages
from pyspark.sql.window import Window               # window functions
from pyspark.sql.functions import row_number        # ranking
from pyspark.ml.feature import HashingTF, IDF       # TF-IDF in Spark
from pyspark.ml.functions import vector_to_array    # convert vector → array

spark = SparkSession.builder.appName("ContentBasedRecommender").getOrCreate()

26/03/18 17:12:18 WARN Utils: Your hostname, rajesh-pc resolves to a loopback address: 127.0.1.1; using 192.168.0.39 instead (on interface wlp1s0)
26/03/18 17:12:18 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/18 17:12:18 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
data_path = "/home/rajesh/CSL7100/Assignment3/ml-latest-small/"

ratings = spark.read.csv(data_path + "ratings.csv", header=True, inferSchema=True)
movies = spark.read.csv(data_path + "movies.csv", header=True, inferSchema=True)

ratings.show(5)
movies.show(5)

+------+-------+------+---------+
|userId|movieId|rating|timestamp|
+------+-------+------+---------+
|     1|      1|   4.0|964982703|
|     1|      3|   4.0|964981247|
|     1|      6|   4.0|964982224|
|     1|     47|   5.0|964983815|
|     1|     50|   5.0|964982931|
+------+-------+------+---------+
only showing top 5 rows

+-------+--------------------+--------------------+
|movieId|               title|              genres|
+-------+--------------------+--------------------+
|      1|    Toy Story (1995)|Adventure|Animati...|
|      2|      Jumanji (1995)|Adventure|Childre...|
|      3|Grumpier Old Men ...|      Comedy|Romance|
|      4|Waiting to Exhale...|Comedy|Drama|Romance|
|      5|Father of the Bri...|              Comedy|
+-------+--------------------+--------------------+
only showing top 5 rows



In [3]:
#TF-IDF on Genres
movies = movies.withColumn("genres", split(col("genres"), "\\|"))   # split genres

hashingTF = HashingTF(inputCol="genres", outputCol="rawFeatures", numFeatures=1000)
tf = hashingTF.transform(movies)   # compute TF

idf = IDF(inputCol="rawFeatures", outputCol="tfidfFeatures")
idf_model = idf.fit(tf)
tfidf = idf_model.transform(tf)   # compute TF-IDF

tfidf = tfidf.withColumn("tfidfArray", vector_to_array(col("tfidfFeatures")))  # vector → array


In [4]:
tfidf.show(3)

+-------+--------------------+--------------------+--------------------+--------------------+--------------------+
|movieId|               title|              genres|         rawFeatures|       tfidfFeatures|          tfidfArray|
+-------+--------------------+--------------------+--------------------+--------------------+--------------------+
|      1|    Toy Story (1995)|[Adventure, Anima...|(1000,[419,780,81...|(1000,[419,780,81...|[0.0, 0.0, 0.0, 0...|
|      2|      Jumanji (1995)|[Adventure, Child...|(1000,[780,812,91...|(1000,[780,812,91...|[0.0, 0.0, 0.0, 0...|
|      3|Grumpier Old Men ...|   [Comedy, Romance]|(1000,[328,419],[...|(1000,[328,419],[...|[0.0, 0.0, 0.0, 0...|
+-------+--------------------+--------------------+--------------------+--------------------+--------------------+
only showing top 3 rows



In [5]:
user_movies = ratings.join(tfidf, on="movieId")   # join ratings with features

# multiply TF-IDF vector with rating (element-wise)
user_movies = user_movies.withColumn(
    "weightedFeatures",
    expr("transform(tfidfArray, x -> x * rating)")
)

In [6]:
from pyspark.sql.functions import sum as spark_sum
from pyspark.sql.functions import posexplode, struct, sort_array

# Step 1: explode features
user_movies_exploded = user_movies.select(
    "userId",
    "rating",
    posexplode("weightedFeatures").alias("idx", "value")
)

# Step 2: sum weighted values per feature
feature_sum = user_movies_exploded.groupBy("userId", "idx").agg(
    spark_sum("value").alias("sumValue")
)

# Step 3: compute total rating per user (IMPORTANT FIX)
rating_sum = ratings.groupBy("userId").agg(
    spark_sum("rating").alias("ratingSum")
)

# Step 4: join
user_profile_norm = feature_sum.join(rating_sum, on="userId")

# Step 5: normalize correctly
user_profile_norm = user_profile_norm.withColumn(
    "normalizedValue",
    col("sumValue") / col("ratingSum")
)

# Step 6: reconstruct vector (correct ordering)
user_profiles = user_profile_norm.groupBy("userId").agg(
    sort_array(collect_list(struct("idx", "normalizedValue"))).alias("features")
)

user_profiles = user_profiles.withColumn(
    "userProfile",
    expr("transform(features, x -> x.normalizedValue)")
)

In [7]:
user_profiles.show(20)

[Stage 14:>                                                         (0 + 5) / 5]

+------+--------------------+--------------------+
|userId|            features|         userProfile|
+------+--------------------+--------------------+
|   148|[{0, 0.0}, {1, 0....|[0.0, 0.0, 0.0, 0...|
|   463|[{0, 0.0}, {1, 0....|[0.0, 0.0, 0.0, 0...|
|   471|[{0, 0.0}, {1, 0....|[0.0, 0.0, 0.0, 0...|
|   496|[{0, 0.0}, {1, 0....|[0.0, 0.0, 0.0, 0...|
|   243|[{0, 0.0}, {1, 0....|[0.0, 0.0, 0.0, 0...|
|   392|[{0, 0.0}, {1, 0....|[0.0, 0.0, 0.0, 0...|
|   540|[{0, 0.0}, {1, 0....|[0.0, 0.0, 0.0, 0...|
|    31|[{0, 0.0}, {1, 0....|[0.0, 0.0, 0.0, 0...|
|   516|[{0, 0.0}, {1, 0....|[0.0, 0.0, 0.0, 0...|
|    85|[{0, 0.0}, {1, 0....|[0.0, 0.0, 0.0, 0...|
|   137|[{0, 0.0}, {1, 0....|[0.0, 0.0, 0.0, 0...|
|   251|[{0, 0.0}, {1, 0....|[0.0, 0.0, 0.0, 0...|
|   451|[{0, 0.0}, {1, 0....|[0.0, 0.0, 0.0, 0...|
|   580|[{0, 0.0}, {1, 0....|[0.0, 0.0, 0.0, 0...|
|    65|[{0, 0.0}, {1, 0....|[0.0, 0.0, 0.0, 0...|
|   458|[{0, 0.0}, {1, 0....|[0.0, 0.0, 0.0, 0...|
|    53|[{0, 0.0}, {1, 0....|[0

In [8]:
#user_profiles.filter(col("userId") == 1).select("userProfile").show(truncate=False)

In [9]:
#Build User Profile
from pyspark.sql.functions import posexplode
from pyspark.sql.functions import struct, sort_array

# explode each vector into (index, value)
user_movies_exploded = user_movies.select(
    "userId",
    "rating",
    posexplode("weightedFeatures").alias("idx", "value")
)

# sum values per index
user_profile_sum = user_movies_exploded.groupBy("userId", "idx").agg(
    expr("sum(value)").alias("sumValue"),
    expr("sum(rating)").alias("ratingSum")
)

# normalize
user_profiles = user_profile_norm.groupBy("userId").agg(
    sort_array(collect_list(struct("idx", "normalizedValue"))).alias("features")
)

user_profiles = user_profiles.withColumn(
    "userProfile",
    expr("transform(features, x -> x.normalizedValue)")
)

In [10]:
user_profiles.show(3)

[Stage 22:>                                                         (0 + 1) / 1]

+------+--------------------+--------------------+
|userId|            features|         userProfile|
+------+--------------------+--------------------+
|   148|[{0, 0.0}, {1, 0....|[0.0, 0.0, 0.0, 0...|
|   463|[{0, 0.0}, {1, 0....|[0.0, 0.0, 0.0, 0...|
|   471|[{0, 0.0}, {1, 0....|[0.0, 0.0, 0.0, 0...|
+------+--------------------+--------------------+
only showing top 3 rows



In [11]:
#Compute similarity

from pyspark.sql.functions import udf
from pyspark.sql.types import DoubleType
import numpy as np

def cosine_similarity(v1, v2):
    v1 = np.array(v1)
    v2 = np.array(v2)
    
    denom = (np.linalg.norm(v1) * np.linalg.norm(v2))
    if denom == 0:
        return 0.0
    
    return float(np.dot(v1, v2) / denom)

cosine_udf = udf(cosine_similarity, DoubleType())

In [12]:
user_movie_sim = user_profiles.crossJoin(tfidf)

user_movie_sim = user_movie_sim.withColumn(
    "similarity",
    cosine_udf(col("userProfile"), col("tfidfArray"))
)

In [13]:
user_movie_sim.show(3)

[Stage 33:>                                                         (0 + 1) / 1]

+------+--------------------+--------------------+-------+--------------------+--------------------+--------------------+--------------------+--------------------+-------------------+
|userId|            features|         userProfile|movieId|               title|              genres|         rawFeatures|       tfidfFeatures|          tfidfArray|         similarity|
+------+--------------------+--------------------+-------+--------------------+--------------------+--------------------+--------------------+--------------------+-------------------+
|   148|[{0, 0.0}, {1, 0....|[0.0, 0.0, 0.0, 0...|      1|    Toy Story (1995)|[Adventure, Anima...|(1000,[419,780,81...|(1000,[419,780,81...|[0.0, 0.0, 0.0, 0...| 0.7506780890166764|
|   148|[{0, 0.0}, {1, 0....|[0.0, 0.0, 0.0, 0...|      2|      Jumanji (1995)|[Adventure, Child...|(1000,[780,812,91...|(1000,[780,812,91...|[0.0, 0.0, 0.0, 0...| 0.6372425002181031|
|   148|[{0, 0.0}, {1, 0....|[0.0, 0.0, 0.0, 0...|      3|Grumpier Old Men ...| 

In [14]:
#Remove watched movie

In [15]:
watched = ratings.select("userId", "movieId")

user_movie_sim = user_movie_sim.join(
    watched,
    on=["userId", "movieId"],
    how="left_anti"
)

In [16]:
watched.show(2)

+------+-------+
|userId|movieId|
+------+-------+
|     1|      1|
|     1|      3|
+------+-------+
only showing top 2 rows



In [29]:
from pyspark.sql.functions import col, expr  # column operations

def recommend_for_user(user_id, user_profiles, tfidf, ratings, movies, top_k=10):

    # target user
    user_vector = user_profiles.filter(col("userId") == user_id)

    # watched movies
    watched_movies = ratings.filter(col("userId") == user_id).select("movieId")

    # similarity computation
    recommendations = user_vector.crossJoin(tfidf)

    # remove watched
    #recommendations = recommendations.join(
    #    watched_movies,
    #    on="movieId",
    #    how="left_anti"
    #)

    # cosine similarity
    recommendations = recommendations.withColumn(
        "similarity",
        expr("""
            aggregate(
                zip_with(userProfile, tfidfArray, (x, y) -> x * y),
                0D,
                (acc, x) -> acc + x
            ) /
            (
                sqrt(aggregate(userProfile, 0D, (acc, x) -> acc + x * x)) *
                sqrt(aggregate(tfidfArray, 0D, (acc, x) -> acc + x * x))
            )
        """)
    )

    # IMPORTANT: assume tfidf already has title
    return recommendations.orderBy(col("similarity").desc()) \
        .select("movieId", "title", "similarity") \
        .limit(top_k)

In [40]:
recs_user1 = recommend_for_user(1, user_profiles, tfidf, ratings, movies)

In [41]:
recs_user1.show(5)

[Stage 195:>                                                        (0 + 1) / 1]

+-------+--------------------+------------------+
|movieId|               title|        similarity|
+-------+--------------------+------------------+
|    546|Super Mario Bros....|0.7961519794705049|
|  26590|G.I. Joe: The Mov...|0.7863512796506192|
|   6350|Laputa: Castle in...|0.7863512796506192|
|  27155|Batman/Superman M...|0.7863512796506192|
|  26340|Twelve Tasks of A...|0.7800686572491826|
+-------+--------------------+------------------+
only showing top 5 rows



In [42]:
from pyspark.sql.functions import collect_list, lit  # aggregation + constant column

recs_user1_df = recs_user1.select("movieId") \
    .agg(collect_list("movieId").alias("recommended")) \
    .withColumn("userId", lit(1))

In [43]:
recs_user1_df.show(truncate=False)

[Stage 206:>                                                        (0 + 1) / 1]

+--------------------------------------------------------------------+------+
|recommended                                                         |userId|
+--------------------------------------------------------------------+------+
|[546, 26590, 27155, 6350, 26340, 51939, 108932, 164226, 79139, 2005]|1     |
+--------------------------------------------------------------------+------+



In [22]:
print(type(recs_user1))

<class 'pyspark.sql.dataframe.DataFrame'>


In [40]:
#evaluate_precision_recall(user_profiles, tfidf, ratings, movies, k=10)

In [32]:
# import required functions
from pyspark.sql.functions import col  # column operations

def evaluate_single_user_k(user_id, user_profiles, tfidf, ratings, movies, k=10):
    """
    Evaluate Precision@K and Recall@K for a single user
    """

    # Step 1: get recommendations (Top-K)
    recs = recommend_for_user(
        user_id,
        user_profiles,
        tfidf,
        ratings,
        movies,
        top_k=k
    )

    # Step 2: extract recommended movie IDs
    recs_list = [row.movieId for row in recs.select("movieId").collect()]

    # Step 3: ground truth (movies rated >= 4)
    relevant = ratings.filter(
        (col("userId") == user_id) & (col("rating") >= 4)
    ).select("movieId")

    relevant_list = [row.movieId for row in relevant.collect()]

    # Step 4: compute intersection
    intersection = set(recs_list).intersection(set(relevant_list))

    # Step 5: compute metrics
    precision = len(intersection) / k if k > 0 else 0
    recall = len(intersection) / len(relevant_list) if relevant_list else 0

    # Step 6: print results
    print(f"User {user_id}")
    print(f"Recommended: {recs_list}")
    print(f"Relevant: {relevant_list}")
    print(f"Intersection: {list(intersection)}")
    print(f"Precision@{k}: {precision:.3f}")
    print(f"Recall@{k}: {recall:.3f}")

    return precision, recall

In [38]:
evaluate_single_user_k(
    1,
    user_profiles,
    tfidf,
    ratings,
    movies,
    k=20
)

[Stage 171:>                                                        (0 + 1) / 1]

User 1
Recommended: [546, 6350, 26590, 27155, 26340, 51939, 108932, 164226, 2005, 3440, 79139, 117646, 558, 71999, 62956, 40339, 52287, 157865, 188301, 7164]
Relevant: [1, 3, 6, 47, 50, 101, 110, 151, 157, 163, 216, 231, 235, 260, 333, 349, 356, 362, 367, 441, 457, 480, 527, 543, 552, 553, 590, 592, 593, 596, 608, 661, 733, 804, 919, 923, 940, 943, 954, 1023, 1024, 1025, 1029, 1031, 1032, 1042, 1049, 1060, 1073, 1080, 1089, 1090, 1092, 1097, 1127, 1136, 1196, 1197, 1198, 1206, 1208, 1210, 1213, 1214, 1220, 1222, 1224, 1226, 1240, 1256, 1265, 1270, 1275, 1278, 1282, 1291, 1298, 1348, 1473, 1500, 1517, 1552, 1573, 1587, 1617, 1620, 1625, 1732, 1777, 1793, 1804, 1805, 1920, 1927, 1954, 1967, 2000, 2005, 2012, 2018, 2028, 2033, 2046, 2048, 2054, 2058, 2078, 2090, 2094, 2096, 2099, 2105, 2115, 2116, 2137, 2139, 2141, 2143, 2161, 2174, 2193, 2268, 2273, 2291, 2329, 2353, 2366, 2387, 2395, 2406, 2427, 2450, 2459, 2470, 2478, 2492, 2502, 2529, 2542, 2571, 2580, 2596, 2616, 2628, 2640, 2641, 26

(0.1, 0.01)

In [44]:
evaluate_single_user_k(
    1,
    user_profiles,
    tfidf,
    ratings,
    movies,
    k=200
)

[Stage 217:>                                                        (0 + 1) / 1]

User 1
Recommended: [546, 6350, 26590, 27155, 26340, 51939, 108932, 164226, 2005, 3440, 79139, 117646, 558, 71999, 62956, 40339, 52287, 157865, 188301, 6624, 7164, 33681, 40851, 70697, 161594, 128944, 2414, 2429, 41569, 673, 130520, 60074, 52462, 7262, 5999, 37830, 58404, 71129, 80083, 177285, 3745, 6016, 55116, 148775, 55167, 485, 3740, 3997, 6539, 53125, 59103, 65588, 87529, 1562, 61818, 5313, 60937, 8961, 26183, 63859, 111146, 152081, 69644, 2625, 27032, 70305, 2041, 2720, 4232, 7345, 26792, 34332, 6503, 8968, 26184, 80219, 134853, 4133, 4275, 5638, 5700, 8782, 32898, 33801, 53453, 57640, 103772, 122902, 122924, 136864, 163056, 166528, 168026, 179819, 2054, 464, 170, 1429, 2835, 4270, 5880, 6709, 7802, 37853, 53972, 64030, 85796, 56152, 187595, 3000, 5069, 139130, 1197, 53140, 3054, 7153, 38294, 55259, 76175, 123553, 156607, 2617, 72165, 4818, 31221, 5657, 6990, 8361, 27618, 48774, 58025, 91500, 117529, 187031, 26504, 459, 4956, 4306, 84637, 92348, 2422, 36397, 1, 2294, 3114, 3754, 

(0.06, 0.06)

In [22]:
spark.stop()